# HERD and HED annotations in Spyglass: DANDI:001888 (Gonzalez-Sulser lab)

Besides their free-text metadata, the converted files carry two kinds of machine-readable annotation. This notebook
reads both back out of a Spyglass database.

- **HERD** (HDMF External Resources Data) links values in the file to terms of external ontologies and registries:
  the species to NCBI Taxonomy, the strain to an RRID, the experimenters to ORCID, the institution to ROR, the
  electrode locations to Uberon anatomy terms, and the three sleep states to the
  [Neuro Behavior Ontology](https://www.ebi.ac.uk/ols4/ontologies/nbo).
- **HED** (Hierarchical Event Descriptors, through the [ndx-hed](https://www.hedtags.org/ndx-hed/) extension)
  describes each sleep state with a controlled vocabulary, in the `HED` column of the sleep scoring table's
  meanings table. The HED schema versions are stored once per file.

The tables are proposed to Spyglass in two PRs. Until they are merged, the environment installs them from the
`gonzalez-sulser-example-v1` tag of the CatalystNeuro Spyglass fork (see the
[Spyglass ingestion docs](../../docs/spyglass_ingestion.md)):

- HERD, [LorenFrankLab/spyglass#1722](https://github.com/LorenFrankLab/spyglass/pull/1722): `OntologyTerm`, a
  catalog of terms shared across sessions, and `ExternalResources`, one row per file, with a `.Reference` part
  holding one row per HERD reference.
- HED, [LorenFrankLab/spyglass#1723](https://github.com/LorenFrankLab/spyglass/pull/1723): `HedMetadata`, the
  schema version and definitions per file, and `HedAnnotation`, one row per HED column of an NWB table, with a
  `.Row` part holding each row's HED string.

These tables mirror the annotations; they do not change the tables that hold the annotated values, such as
`Subject`, `BrainRegion` or `SleepStateScoring`. Joining by value relates them, as the sections below show.

**Running it:** this notebook runs in the `spyglass-gonzalez-sulser-lab-env` conda env inside WSL2, created with `../bootstrap_wsl.sh`. The database is the `spyglass-db` Docker container on port 3306; see `../docker-compose.yml`, `../dj_local_conf.json` and the [Spyglass ingestion docs](../../docs/spyglass_ingestion.md). Ingest the files with `../insert_full_session.py` first.

In [1]:
from pathlib import Path

import datajoint as dj

# Load the connection config BEFORE importing spyglass: spyglass declares its schemas
# at import time and would otherwise connect with default settings (including TLS,
# which the local Docker MySQL doesn't use).
dj.config.load(str(Path("..") / "dj_local_conf.json"))
dj.conn(use_tls=False)

[2026-11-19 11:30:41,957][INFO]: DataJoint is configured from /mnt/c/Users/amtra/CatalystNeuro/gonzalez-sulser-lab-to-nwb/src/gonzalez_sulser_lab_to_nwb/hristova_2025/spyglass/dj_local_conf.json


[2026-11-19 11:30:41,967][INFO]: DataJoint 0.14.10 connected to root@127.0.0.1:3306


DataJoint connection (connected) root@127.0.0.1:3306

In [2]:
import json

import pandas as pd
from hed import HedString, load_schema_version
from hed.validator import HedValidator
from IPython.display import HTML, display

from spyglass.common import (
    BrainRegion,
    ElectrodeGroup,
    ExternalResources,
    HedAnnotation,
    HedMetadata,
    Nwbfile,
    OntologyTerm,
    Session,
    SleepStateScoring,
    Subject,
)

pd.set_option("display.max_colwidth", 80)
GENOTYPE_LABELS = {"LE-Grin2bem1Mcwi": "Het", "WT": "WT"}

sessions = (
    pd.DataFrame((Session * Subject).fetch("nwb_file_name", "subject_id", "genotype", as_dict=True))
    .assign(genotype=lambda df: df["genotype"].map(GENOTYPE_LABELS))
    .sort_values("subject_id")
    .reset_index(drop=True)
)
example = sessions[sessions["subject_id"] == "GRIN2B-424"].iloc[0]
example_key = {"nwb_file_name": example["nwb_file_name"]}

## 1. HERD: links to external ontologies

### The terms

`OntologyTerm` holds every term any file refers to, once. Each `entity_id` is a compact identifier,
`PREFIX:local_id`, and `entity_uri` resolves it. The prefix names the ontology or registry; the list below links
each one present to its [Bioregistry](https://bioregistry.io/) page.

In [3]:
terms = OntologyTerm.fetch(format="frame").reset_index()
terms["prefix"] = terms["entity_id"].str.split(":").str[0]
display(terms)

ONTOLOGY_INFO = {
    "NBO": ("Neuro Behavior Ontology", "behavioral processes, here the sleep states"),
    "NCBITaxon": ("NCBI Taxonomy", "taxonomic identifiers, here the species"),
    "RRID": ("Research Resource Identifiers", "research resources such as animal strains"),
    "orcid": ("ORCID", "researcher identifiers, here the experimenters"),
    "ror": ("Research Organization Registry", "institution identifiers"),
    "UBERON": ("Uberon", "cross-species anatomy, here the cortical areas and the neck muscle"),
}
items = []
for prefix, n_terms in terms["prefix"].value_counts().sort_index().items():
    name, description = ONTOLOGY_INFO.get(prefix, (prefix, ""))
    url = f"https://bioregistry.io/registry/{prefix.lower()}"
    items.append(
        f'<li><a href="{url}" target="_blank"><b>{name}</b></a> (<code>{prefix}</code>, {n_terms} terms): {description}</li>'
    )
display(HTML("<ul>" + "".join(items) + "</ul>"))

,entity_id,entity_uri,prefix
0,NBO:0000066,http://purl.obolibrary.org/obo/NBO_0000066,NBO
1,NBO:0000159,http://purl.obolibrary.org/obo/NBO_0000159,NBO
2,NBO:0000161,http://purl.obolibrary.org/obo/NBO_0000161,NBO
3,NCBITaxon:10116,http://purl.obolibrary.org/obo/NCBITaxon_10116,NCBITaxon
4,orcid:0000-0001-8868-5478,https://orcid.org/0000-0001-8868-5478,orcid
5,orcid:0000-0001-8938-2268,https://orcid.org/0000-0001-8938-2268,orcid
6,orcid:0000-0002-2105-8725,https://orcid.org/0000-0002-2105-8725,orcid
7,orcid:0000-0002-4256-9639,https://orcid.org/0000-0002-4256-9639,orcid
8,orcid:0000-0002-4373-8262,https://orcid.org/0000-0002-4373-8262,orcid
9,orcid:0000-0002-9874-2939,https://orcid.org/0000-0002-9874-2939,orcid


### One session's references

`ExternalResources` has one row per file, with its number of references: 27 in every file. `fetch_references_dataframe()`
returns the references of the restricted sessions joined with their terms. Each row names the annotated NWB object
(`object_id`, `object_type`), where in it the value sits (`relative_path`), the value itself (`key`) and the term it
links to.

Two references have `object_type` `VectorData`: a column of a table. The object id says which one, and the file
says which table and column that id is. The helper below opens the file through `Nwbfile` and names the target of
each reference.

In [4]:
def describe_targets(key, object_ids):
    """'table.column' (or the object's name) for each NWB object id, read from the file."""
    nwbfile = (Nwbfile & key).fetch_nwb()[0]
    try:
        names = {}
        for object_id in object_ids:
            obj = nwbfile.objects[object_id]
            names[object_id] = f"{obj.parent.name}.{obj.name}" if type(obj).__name__ == "VectorData" else obj.name
        return names
    finally:
        (Nwbfile & key).close_nwb()


references = (ExternalResources & example_key).fetch_references_dataframe().sort_values("reference_index")
references["target"] = references["object_id"].map(describe_targets(example_key, references["object_id"].unique()))
print(f"{example['subject_id']}: {len(references)} references")
references[["reference_index", "object_type", "target", "relative_path", "key", "entity_id"]].reset_index(drop=True)

GRIN2B-424: 27 references


,reference_index,object_type,target,relative_path,key,entity_id
0,0,VectorData,sleep_state_meanings.value,,Wake,NBO:0000066
1,1,VectorData,sleep_state_meanings.value,,NREM,NBO:0000161
2,2,VectorData,sleep_state_meanings.value,,REM,NBO:0000159
3,3,Subject,subject,species,Rattus norvegicus,NCBITaxon:10116
4,4,Subject,subject,strain,Long-Evans,RRID:RGD_2308852
5,5,VectorData,electrodes.location,,"primary somatosensory cortex, trunk region (S1-Tr)",UBERON:0008933
6,6,VectorData,electrodes.location,,"secondary motor cortex, frontal association area (M2-FrA)",UBERON:0016634
7,7,VectorData,electrodes.location,,"secondary motor cortex, anterior (M2-Ant)",UBERON:0016634
8,8,VectorData,electrodes.location,,"primary motor cortex, anterior (M1-Ant)",UBERON:0001384
9,9,VectorData,electrodes.location,,"secondary visual cortex, mediolateral area (V2-ML)",UBERON:0022232


### Linking references to Spyglass's own tables

A reference points at a value in the NWB file, and Spyglass's own tables hold many of the same values. Joining on
the value relates them:

- **Species.** `Subject.species` holds the species name; the reference on the subject's `species` links it to NCBI
  Taxonomy. The strain has no column in `Subject`, so its RRID is only in the references.
- **Electrode groups.** `ElectrodeGroup` points to `BrainRegion`, whose `region_name` is the group's location
  (`cerebral cortex` and `trapezius muscle`); the references on the groups' `location` link them to Uberon.
- **Experimenters.** `Session.Experimenter` lists the experimenters as `First Last`; the references on the file's
  `general/experimenter` use `Last, First` and link them to ORCID. 11 of the 13 experimenters have an ORCID in the
  files, so two rows below have no term.
- **Sleep states.** `SleepStateScoring.Epoch.sleep_state` holds each epoch's label; the references on the
  `value` column of the meanings table link Wake, NREM and REM to the Neuro Behavior Ontology.

The cortical area of each EEG screw is linked too, through the `location` column of the file's electrodes table.
Spyglass's `Electrode` has no column for it (see the [EEG and EMG notebook](spyglass_eeg_emg_tutorial.ipynb)), so
that join goes through the file.

In [5]:
all_references = ExternalResources().fetch_references_dataframe()


def references_on(object_type, relative_path=""):
    return all_references.query("object_type == @object_type and relative_path == @relative_path")[
        ["nwb_file_name", "key", "entity_id", "entity_uri"]
    ]


species = (
    pd.DataFrame((Session * Subject).fetch("nwb_file_name", "species", as_dict=True))
    .merge(references_on("Subject", "species"), left_on=["nwb_file_name", "species"], right_on=["nwb_file_name", "key"])
    .groupby(["species", "entity_id"])["nwb_file_name"].nunique().rename("n_sessions").reset_index()
)
display(species)

group_regions = (
    pd.DataFrame((ElectrodeGroup * BrainRegion).fetch("nwb_file_name", "electrode_group_name", "region_name", as_dict=True))
    .merge(
        references_on("ElectrodeGroup", "location"),
        left_on=["nwb_file_name", "region_name"], right_on=["nwb_file_name", "key"], how="left",
    )
    .groupby(["electrode_group_name", "region_name", "entity_id"])["nwb_file_name"].nunique().rename("n_sessions").reset_index()
)
display(group_regions)

experimenters = pd.DataFrame((Session.Experimenter & example_key).fetch("lab_member_name", as_dict=True))
orcid = references_on("NWBFile", "general/experimenter").query("nwb_file_name == @example_key['nwb_file_name']").copy()
orcid["lab_member_name"] = orcid["key"].map(lambda name: " ".join(reversed([part.strip() for part in name.split(",", 1)])))
experimenters.merge(orcid[["lab_member_name", "entity_id", "entity_uri"]], on="lab_member_name", how="left")

,species,entity_id,n_sessions
0,Rattus norvegicus,NCBITaxon:10116,24


,electrode_group_name,region_name,entity_id,n_sessions
0,EEGArray,cerebral cortex,UBERON:0000956,24
1,EMGArray,trapezius muscle,UBERON:0002380,24


,lab_member_name,entity_id,entity_uri
0,Adrian Ocampo-Garces,orcid:0000-0002-9874-2939,https://orcid.org/0000-0002-9874-2939
1,Alejandro Bassi,orcid:0000-0002-4373-8262,https://orcid.org/0000-0002-4373-8262
2,Alfredo Gonzalez-Sulser,orcid:0000-0003-3494-4029,https://orcid.org/0000-0003-3494-4029
3,Andrew Sutherland,orcid:0009-0003-2874-2591,https://orcid.org/0009-0003-2874-2591
4,Anjanette P. Harris,orcid:0000-0001-8938-2268,https://orcid.org/0000-0001-8938-2268
5,Ingrid Buller-Peralta,NaN,NaN
6,Javier Escudero,orcid:0000-0002-2105-8725,https://orcid.org/0000-0002-2105-8725
7,Katerina Hristova,orcid:0009-0000-4079-8676,https://orcid.org/0009-0000-4079-8676
8,Mehmet Taskiran,orcid:0000-0003-0061-6908,https://orcid.org/0000-0003-0061-6908
9,Melissa C. M. Fasol,orcid:0000-0001-8868-5478,https://orcid.org/0000-0001-8868-5478


In [6]:
# Cortical areas of the EEG screws: the electrodes table's location column, read from the file
nwbfile = (Nwbfile & example_key).fetch_nwb()[0]
file_electrodes = nwbfile.electrodes.to_dataframe()[["group_name", "channel_name", "location"]]
location_column_id = nwbfile.electrodes["location"].object_id
(Nwbfile & example_key).close_nwb()

location_references = all_references.query(
    "nwb_file_name == @example_key['nwb_file_name'] and object_id == @location_column_id"
)[["key", "entity_id", "entity_uri"]]
file_electrodes.merge(location_references, left_on="location", right_on="key", how="left").drop(columns="key")

,group_name,channel_name,location,entity_id,entity_uri
0,EEGArray,S1_Tr_R,"primary somatosensory cortex, trunk region (S1-Tr)",UBERON:0008933,http://purl.obolibrary.org/obo/UBERON_0008933
1,EEGArray,M2_Fra_R,"secondary motor cortex, frontal association area (M2-FrA)",UBERON:0016634,http://purl.obolibrary.org/obo/UBERON_0016634
2,EEGArray,M2_anterior_R,"secondary motor cortex, anterior (M2-Ant)",UBERON:0016634,http://purl.obolibrary.org/obo/UBERON_0016634
3,EEGArray,M1_anterior_R,"primary motor cortex, anterior (M1-Ant)",UBERON:0001384,http://purl.obolibrary.org/obo/UBERON_0001384
4,EEGArray,V2_ML_R,"secondary visual cortex, mediolateral area (V2-ML)",UBERON:0022232,http://purl.obolibrary.org/obo/UBERON_0022232
5,EEGArray,V1_M_R,"primary visual cortex, monocular area (V1-M)",UBERON:0002436,http://purl.obolibrary.org/obo/UBERON_0002436
6,EEGArray,S1Hl_S1Fl_R,"primary somatosensory cortex, hindlimb and forelimb regions (S1-HL/FL)",UBERON:0008933,http://purl.obolibrary.org/obo/UBERON_0008933
7,EEGArray,V1_M_L,"primary visual cortex, monocular area (V1-M)",UBERON:0002436,http://purl.obolibrary.org/obo/UBERON_0002436
8,EEGArray,V2_ML_L,"secondary visual cortex, mediolateral area (V2-ML)",UBERON:0022232,http://purl.obolibrary.org/obo/UBERON_0022232
9,EEGArray,S1Hl_S1Fl_L,"primary somatosensory cortex, hindlimb and forelimb regions (S1-HL/FL)",UBERON:0008933,http://purl.obolibrary.org/obo/UBERON_0008933


### Sleep states and their terms

The three sleep-state labels are linked to Neuro Behavior Ontology terms. Since `SleepStateScoring.Epoch` stores
the label of every 5-second epoch, joining on the label gives each epoch its term, in the database. Below, the
epochs of every session are counted per term.

In [7]:
# The sleep-state references are the ones whose keys are the labels used in SleepStateScoring.Epoch
labels = set(SleepStateScoring.Epoch.fetch("sleep_state"))
state_terms = all_references[all_references["key"].isin(labels)][["key", "entity_id", "entity_uri"]].drop_duplicates()
state_terms = state_terms.rename(columns={"key": "sleep_state"}).reset_index(drop=True)
display(state_terms)

counts = (
    dj.U("sleep_state").aggr(SleepStateScoring.Epoch, n_epochs="count(*)", hours="sum(stop_time - start_time) / 3600")
    .fetch(format="frame")
    .reset_index()
    .merge(state_terms, on="sleep_state")
)
counts

,sleep_state,entity_id,entity_uri
0,Wake,NBO:0000066,http://purl.obolibrary.org/obo/NBO_0000066
1,REM,NBO:0000159,http://purl.obolibrary.org/obo/NBO_0000159
2,NREM,NBO:0000161,http://purl.obolibrary.org/obo/NBO_0000161


,sleep_state,n_epochs,hours,entity_id,entity_uri
0,Wake,535885,744.284722,NBO:0000066,http://purl.obolibrary.org/obo/NBO_0000066
1,NREM,262330,364.347222,NBO:0000161,http://purl.obolibrary.org/obo/NBO_0000161
2,REM,31225,43.368056,NBO:0000159,http://purl.obolibrary.org/obo/NBO_0000159


### Querying by term

Because the terms are database rows, ontology identifiers work as query keys. The query below selects the NREM
epochs (`NBO:0000161`, non-REM sleep) of every session through the term, not through the lab's label: in a database
holding several datasets that label their states differently, the same query selects across them.

In [8]:
NREM_SLEEP = "NBO:0000161"
nrem_labels = (ExternalResources.Reference & {"entity_id": NREM_SLEEP}).fetch("key")
nrem_epochs = SleepStateScoring.Epoch & [{"sleep_state": label} for label in set(nrem_labels)]
print(f"{len(nrem_epochs)} epochs in {len(dj.U('nwb_file_name') & nrem_epochs)} sessions are scored as {NREM_SLEEP} ({set(nrem_labels)})")

262330 epochs in 24 sessions are scored as NBO:0000161 ({'NREM'})


### HERD completeness checks

Every session should annotate the species, strain, experimenters, institution, both electrode groups' locations,
the electrodes' locations and the three sleep states.

In [9]:
EXPECTED_PATHS = {
    ("Subject", "species"),
    ("Subject", "strain"),
    ("NWBFile", "general/experimenter"),
    ("NWBFile", "general/institution"),
    ("ElectrodeGroup", "location"),
    ("VectorData", ""),  # the electrodes' location column and the sleep-state labels
}
for name in sessions["nwb_file_name"]:
    refs = all_references[all_references["nwb_file_name"] == name]
    assert len(refs) == 27, f"{name}: {len(refs)} references"
    assert set(zip(refs["object_type"], refs["relative_path"])) == EXPECTED_PATHS, f"{name}: annotated values"
    assert set(refs["key"]) >= {"Wake", "NREM", "REM"}, f"{name}: sleep states"
    assert set(refs.query("object_type == 'ElectrodeGroup'")["key"]) == {"cerebral cortex", "trapezius muscle"}

print(f"All HERD checks passed for {len(sessions)} session(s).")

All HERD checks passed for 24 session(s).


## 2. HED: the sleep states in a controlled vocabulary

### Schema versions

`HedMetadata` has one row per file with the HED schema versions the tags use: the standard schema 8.4.0 and the
[SCORE library](https://www.hedtags.org/hed-schemas/) 2.1.0 (prefix `sc:`), which has terms for EEG sleep stages.
The files have no HED definitions.

In [10]:
hed_metadata = HedMetadata.fetch(format="frame").reset_index()
print("HED schema versions:", sorted(hed_metadata["hed_schema_version"].unique()))
print("Files with definitions:", (hed_metadata["definitions"] != "").sum())

hed_schema = load_schema_version(json.loads(hed_metadata["hed_schema_version"].iloc[0]))

HED schema versions: ['["8.4.0", "sc:score_2.1.0"]']
Files with definitions: 0


### The annotations

`HedAnnotation` has one row per HED column found in a file: here the `HED` column of `sleep_state_meanings`, the
meanings table of the sleep scoring table, with one `.Row` per state. The database stores the HED string and the
row index; the state each row describes is in the meanings table's `value` column, read from the file through
`table_object_id`.

In [11]:
annotations = HedAnnotation().fetch_annotations_dataframe()
print(annotations.groupby(["table_name", "column_name"])["nwb_file_name"].nunique().rename("n_sessions"))

table_object_id = (HedAnnotation & example_key).fetch1("table_object_id")
nwbfile = (Nwbfile & example_key).fetch_nwb()[0]
meanings = nwbfile.objects[table_object_id].to_dataframe()[["value", "meaning"]]
(Nwbfile & example_key).close_nwb()

example_annotations = annotations[annotations["nwb_file_name"] == example_key["nwb_file_name"]]
meanings = meanings.reset_index(drop=True).join(example_annotations.set_index("row_index")["hed_string"])
meanings

table_name            column_name
sleep_state_meanings  HED            24
Name: n_sessions, dtype: int64


,value,meaning,hed_string
0,Wake,Wakefulness.,Awake
1,NREM,Non-rapid eye movement (NREM) sleep.,Asleep
2,REM,Rapid eye movement (REM) sleep.,"Asleep, sc:Sleep-stage-REM"


### Querying the epochs by tag

With the HED string of each state, every scored epoch gets tags. [hedtools](https://pypi.org/project/hedtools/),
installed with ndx-hed, parses the strings against the schema. Below, the time asleep in each session is the
duration of the epochs whose state has the `Asleep` tag; it matches the NREM plus REM time.

In [12]:
state_tags = {
    row.value: {str(tag) for tag in HedString(row.hed_string, hed_schema).get_all_tags()}
    for row in meanings.itertuples()
}
for state, tags in state_tags.items():
    print(f"{state}: {sorted(tags)}")

asleep_states = [state for state, tags in state_tags.items() if "Asleep" in tags]
per_state = (
    dj.U("nwb_file_name", "sleep_state")
    .aggr(SleepStateScoring.Epoch, hours="sum(stop_time - start_time) / 3600")
    .fetch(format="frame")
    .reset_index()
    .pivot(index="nwb_file_name", columns="sleep_state", values="hours")
)
per_state["asleep (HED)"] = per_state[asleep_states].sum(axis=1)
assert (per_state["asleep (HED)"] - per_state["NREM"] - per_state["REM"]).abs().max() < 1e-9
sessions.set_index("nwb_file_name")[["subject_id", "genotype"]].join(per_state.round(2)).reset_index(drop=True)

Wake: ['Awake']
NREM: ['Asleep']
REM: ['Asleep', 'sc:Sleep-stage-REM']


,subject_id,genotype,NREM,REM,Wake,asleep (HED)
0,GRIN2B-129,Het,15.990278,0.502778,31.506944,16.493056
1,GRIN2B-130,Het,14.884722,1.813889,31.301389,16.698611
2,GRIN2B-131,Het,16.579167,2.195833,29.225,18.775
3,GRIN2B-132,WT,12.258333,1.241667,34.5,13.5
4,GRIN2B-137,Het,14.465278,1.0,32.534722,15.465278
5,GRIN2B-138,Het,15.236111,1.434722,31.329167,16.670833
6,GRIN2B-139,Het,14.129167,1.259722,32.611111,15.388889
7,GRIN2B-140,WT,13.725,1.879167,32.395833,15.604167
8,GRIN2B-227,Het,15.383333,2.279167,30.3375,17.6625
9,GRIN2B-228,Het,14.275,1.418056,32.306944,15.693056


### Validation and checks

`HedValidator` checks each annotation against the schemas; an empty list of issues means it is valid. The checks
then assert, for every session, that the file has HED metadata and that the meanings table's `HED` column was
ingested with one valid string per state.

In [13]:
validator = HedValidator(hed_schema)
annotations["n_issues"] = annotations["hed_string"].map(
    lambda s: len(validator.validate(HedString(s, hed_schema), allow_placeholders=False))
)

columns = HedAnnotation.fetch(format="frame").reset_index()
for name in sessions["nwb_file_name"]:
    assert name in set(hed_metadata["nwb_file_name"]), f"{name}: no HedMetadata"
    session_columns = columns[columns["nwb_file_name"] == name]
    assert list(zip(session_columns["table_name"], session_columns["column_name"])) == [("sleep_state_meanings", "HED")]
    rows = annotations[annotations["nwb_file_name"] == name]
    assert list(rows.sort_values("row_index")["hed_string"]) == list(meanings["hed_string"]), f"{name}: HED strings"
    assert (rows["n_issues"] == 0).all(), f"{name}: HED validation issues"

print(f"All HED checks passed for {len(sessions)} session(s).")

All HED checks passed for 24 session(s).


## Summary

- **HERD:** every file links its species, strain, experimenters, institution, electrode locations and sleep states
  to external terms. In Spyglass the terms are shared rows of `OntologyTerm` and each file's 27 references are rows
  of `ExternalResources.Reference`, so they join to `Subject`, `BrainRegion`, `Session.Experimenter` and
  `SleepStateScoring.Epoch` by value, and work as query keys across sessions. The per-electrode cortical areas join
  through the file, because Spyglass's `Electrode` has no column for them.
- **HED:** every file describes Wake, NREM and REM with HED tags from the standard schema and the SCORE library.
  `HedMetadata` keeps the schema versions and `HedAnnotation` the strings, which validate and tag each scored epoch.